## Mistakes were made...

This is a collection of code snippets for troubleshooting and editing Dataverse collections via API. They may or may not be helpful for curating a large collection of datasets. It's messy and unmaintained. Use at your own risk! More information available at: 
- Dataverse API guide: https://guides.dataverse.org/en/latest/api/index.html
- pyDataverse documentation: https://pydataverse.readthedocs.io/en/latest/

### 1. Move datasets into different dv collection

`for doi in list_of_dois:`
    `POST http://$SERVER/api/datasets/$id/move/$alias`

In [ ]:
import requests

In [ ]:
base_url = "https://dataverse.harvard.edu"
api_token = {api_token}
alias = 'levy_photos_1993'

In [ ]:
doi_text_file = open('1993/1993_dois.txt', 'r')
doi_text = doi_text_file.readlines()
doi_text

In [ ]:
dois_stripped = []
for t in doi_text:
    dois_stripped.append(t.rstrip())

In [ ]:
dois_stripped

In [ ]:
headers = {'X-Dataverse-key': api_token,}

for i in dois_stripped:
    url_dataset_move = '%s/api/datasets/:persistentId/move/levy_photos_1993?persistentId=%s' % (base_url, i)
    r = requests.post(url_dataset_move, headers=headers)
    print(r.json())

### 2. Edit Dataset Metadata

First need to create json file with field(s) to update. Ex. uses "Grant information"

`{
    "fields": [
		{
            "typeName": "grantNumber",
            "multiple": true,
            "typeClass": "compound",
            "value": [
                {
                    "grantNumberAgency": {
                        "typeName": "grantNumberAgency",
                        "multiple": false,
                        "typeClass": "primitive",
                        "value": "National Science Foundation"
                    }
                }
            ]
        }
    ]
}`

**Strategy:** `for DOI in list_of_dois: PUT api_call --upload metadata_edit.json`

In [ ]:
import requests
import json
from time import sleep

In [ ]:
base_url = "https://dataverse.harvard.edu"
api_token = "xxxx"
alias = 'levy_photos_2012'

headers = {'X-Dataverse-key': api_token, 'Content-Type' : 'application/json'}

with open("good_metadata.json", "rb") as f:
    dat = json.load(f)
    
doi = "doi:10.7910/DVN/XXXXXX"
url_dataset_metadata_edit = '%s/api/datasets/:persistentId/editMetadata/?persistentId=%s' % (base_url, doi)
r = requests.put(url_dataset_metadata_edit, headers=headers, data=json.dumps(dat))
print(r.json())

In [ ]:
#get json of good dataset metadata
doi = 'doi:10.7910/DVN/W0QNTN'
get_json_url = "%s/api/datasets/:persistentId/?persistentId=%s" % (base_url, doi)
r = requests.get(get_json_url, headers=headers)
json_dict = r.json()
jsonString = json.dumps(json_dict)
jsonFile = open("good_metadata.json", "w")
jsonFile.write(jsonString)
jsonFile.close()

#manually create template for updating

In [ ]:
doi_text_file = open('2012/2012_dois5.txt', 'r')
doi_text = doi_text_file.readlines()
dois_stripped = []
for t in doi_text:
    dois_stripped.append(t.rstrip())
dois_stripped

In [ ]:
with open("good_metadata.json", "rb") as f:
    dat = json.load(f)
    
print(dat)

In [ ]:
doi = "doi:10.7910/DVN/QLDYXT"
url_dataset_metadata_edit = '%s/api/datasets/:persistentId/editMetadata/?persistentId=%s' % (base_url, doi)
r = requests.put(url_dataset_metadata_edit, headers=headers, data=json.dumps(dat))
print(r.json())

In [ ]:
for i in dois_stripped:
    url_dataset_metadata_edit = '%s/api/datasets/:persistentId/editMetadata/?persistentId=%s&replace=true' % (base_url, i)
    r = requests.put(url_dataset_metadata_edit, headers=headers, data=json.dumps(dat))
    print(r.json()['status'], "Sleeping...")
    sleep(10)

In [ ]:
r.text

### 3. Identifying datasets without files

Strategy: 
1. For each DOI, list files
2. Find objects in list with empty `data` values
3. Match `'data' = []` to DOI by line

In [ ]:
import pandas as pd
import requests
import json

In [ ]:
doi_text_file = open('2012/2012_dois.txt', 'r') ##Need to run for 1985, but need to collectino list of dois first
doi_text = doi_text_file.readlines()
doi_text

In [ ]:
dois_stripped = []
for t in doi_text:
    dois_stripped.append(t.rstrip())

In [ ]:
base_url = "https://dataverse.harvard.edu"
api_token = {api_token}
alias = 'levy_photos_2012'

In [ ]:
test_set = dois_stripped[0:5]
test_set.append('doi:10.7910/DVN/1PNWNY')
test_set

In [ ]:
all_dataset_files = []
headers = {'X-Dataverse-key': api_token,}

for doi in dois_stripped:
    url_request = "%s/api/datasets/:persistentId/versions/:draft/files/?persistentId=%s" % (base_url, doi)
    r = requests.get(url_request, headers=headers)
    all_dataset_files.append(json.loads(r.text))

In [ ]:
#find objects with empty "data" values
for i in all_dataset_files:
    print(i['data']==[])

Copy into txt file and match TRUE lines to same line in DOI file to find DOI with missing files

### 4. GET dataset PIDs and upload files

`dataset_id_2_pid` list is incomplete because your computer fell asleep during the dataset creation or the API requets timed out or something else happened. 
1. Get PIDs for all datasets in a dataverse
2. Make sure that list of dataset PIDs is identical to datafile metadata csv
3. Upload file to each dataset

In [ ]:
import requests
import json
import os

In [ ]:
base_url = "https://dataverse.harvard.edu"
api_token = {api_token}
alias = 'levy_photos_2012'

In [ ]:
headers = {'X-Dataverse-key': api_token,}

url_request = "%s/api/dataverses/%s/contents" % (base_url, alias)
r = requests.get(url_request, headers=headers)
json = r.json()['data']

In [ ]:
pids = [i['identifier'] for i in json]
print(pids)

In [ ]:
def prepend(l, s):
      
    # Using format()
    s += '{0}'
    l = [s.format(i) for i in l]
    return(l)
  
# Driver function
l = pids
s = 'doi:10.7910/'
pids = prepend(l, s)
print(pids)

In [ ]:
#save list of dois to file
with open("2012/2012_dois.txt", "w") as f: 
    for doi in pids: 
        f.write(doi + '\n')

In [ ]:
dois = dict(enumerate(pids, start = 1))
print(dois)

In [ ]:
d = {str(key): str(value) for key, value in dois.items()}
print(d)

In [ ]:
#read in datafiles csv
from pyDataverse.utils import read_csv_as_dicts
df_data = read_csv_as_dicts('1992/levy_datafiles_1992.csv', encoding = 'utf-8-sig')
print(df_data)

In [ ]:
#turn data files dicts into Datafile objects
from pyDataverse.models import Datafile
df_lst = []
for df in df_data:
    df_obj = Datafile()
    df_obj.set(df)
    df_lst.append(df_obj)

In [ ]:
#test API connection
from pyDataverse.api import NativeApi
api = NativeApi(base_url, api_token)
resp = api.get_info_version()
resp.json()

In [ ]:
#upload files with dois instead of dataset_id_2_pid
for df in df_lst:
    pid = d[df.get()['org.dataset_id']]
    filename = os.path.join('/Users/katherinemika/Desktop/curation/leon_levy/1992', df.get()['org.filename'])
    df.set({"pid": pid, "filename": filename})
    resp = api.upload_datafile(pid, filename, df.json())

### 5. Move dataverse collection into other dataverse collection

In [ ]:
import requests
import json

In [ ]:
api_token = {api_token}
server = "https://dataverse.harvard.edu"

In [ ]:
headers = {'X-Dataverse-key': api_token,}

url_dataverse_move = '%s/api/dataverses/levy_photos_2012/move/levy_photos' % (server)
r = requests.post(url_dataverse_move, headers=headers)
print(r.json())

### 6. Find missing images from collection

In [ ]:
import pandas as pd
import requests

`get all dois in collection`

`for i in dois:
    get title[i]`

Match filenames in collection with filenames in OCHRE spreadsheet

In [ ]:
api_token = {api_token}
base_url = "https://dataverse.harvard.edu"
ID = "levy_photos_2012"

headers = {'X-Dataverse-key': api_token,}

In [ ]:
get_contents_url = '%s/api/dataverses/%s/contents' % (base_url, ID)
r = requests.get(get_contents_url, headers=headers)
contents = r.json()['data']

In [ ]:
contents[0]['identifier']

In [ ]:
pids = [i['identifier'] for i in contents]
print(pids)

In [ ]:
def prepend(l, s):
      
    # Using format()
    s += '{0}'
    l = [s.format(i) for i in l]
    return(l)
  
# Driver function
l = pids
s = 'doi:10.7910/'
pids = prepend(l, s)
print(pids)

In [ ]:
title = []

for p in pids:
    get_dataset_metadata_url = "%s/api/datasets/:persistentId/?persistentId=%s" % (base_url, p)
    r = requests.get(get_dataset_metadata_url, headers=headers)
    if(r.json()['status']== "OK"):
        title.append(r.json()['data']['latestVersion']["metadataBlocks"]["citation"]["fields"][0]["value"])
        continue

pids_zip = list(zip(pids, title))
doi_df = pd.DataFrame(pids_zip, columns=['pid','title'])

In [ ]:
doi_df

In [ ]:
filename = title
filename

In [ ]:
full_2012 = pd.read_csv("2012_metadata.csv")
full_2012.head()

In [ ]:
remaining_2012 = full_2012[~full_2012.filename.isin(title)]

In [ ]:
remaining_2012 = remaining_2012.reset_index()
remaining_2012

In [ ]:
remaining_2012.to_csv("remaining_2012.csv")

In [ ]:
valid_2012 = full_2012[full_2012.filename.isin(title)]
valid_2012.to_csv("valid_2012.csv")

### 7. Batch edit metadata from csv file

1. Create json template
    - GET json representation of a correct dataset
    - Transform into generic template

2. Create json files for each dataset 
`for row in csv:
    json[value1] = csv[value1]
    json[value2] = csv[value2]
    ...
    save json as new file`
 
3. Update metadata using json template

In [ ]:
import pandas as pd
import json
import requests
import itertools

api_token = {api_token}
base_url = "https://dataverse.harvard.edu"

headers = {'X-Dataverse-key': api_token,}

In [ ]:
#get metadata to create template
get_dataset_metadata_url = "%s/api/datasets/:persistentId/?persistentId=doi:10.7910/DVN/3BVWHY" % (base_url)
r = requests.get(get_dataset_metadata_url, headers=headers)
template = r.json()['data']
writeFile = open('template.json', 'w')
json.dump(template, writeFile)
writeFile.close()

Manually edit template with standard fields.

In [ ]:
with open("template.json", "r") as f:
    temp = f.read()

json_template = json.loads(temp)

In [ ]:
json_template['fields'][0]['value']

In [ ]:
json_template['fields'][4]['value'][0]['dsDescriptionValue']['value']

Create metadata spreadsheet for unique cols (title, ds.description, & dv.alternativeURL

In [ ]:
valid_2012['dv.title']=valid_2012["filename"]
valid_2012.head()

In [ ]:
valid_2012['dv.dsDescription'] = "Link to OCHRE database: <a href = \''" + valid_2012['ochre_link'] + '\'>' + valid_2012['ochre_link']
valid_2012.head()

In [ ]:
for index, row in valid_2012.iterrows():
    print(row['dv.title'])

In [ ]:
for index, row in valid_2012.iterrows():
    json_template['fields'][0]['value'] = row['dv.title']
    print(json_template['fields'][0]['value'])

In [ ]:
for index, row in valid_2012.iterrows():
    json_template['fields'][0]['value'] = row['dv.title']
    json_template['fields'][4]['value'][0]['dsDescriptionValue']['value'] = row['dv.dsDescription']
    json_template['fields'][1]['value'] = row['ochre_link']
    writef = open('2012/%s.json' % str(row['dv.title']), 'w')
    json.dump(json_template, writef)
    writef.close()

fix extra .jpg in filename: `for i in *; do mv "$i" "`echo $i | sed 's/.jpg//'`"; done`

In [ ]:
#get pids from DV
ID = "levy"
get_contents_url = '%s/api/dataverses/%s/contents' % (base_url, ID)
r = requests.get(get_contents_url, headers=headers)
contents = r.json()['data']

In [ ]:
contents[0]['identifier']

In [ ]:
pids = [i['identifier'] for i in contents]
print(pids)

In [ ]:
def prepend(l, s):
      
    # Using format()
    s += '{0}'
    l = [s.format(i) for i in l]
    return(l)
  
# Driver function
l = pids
s = 'doi:10.70122/'
pids = prepend(l, s)
print(pids)

In [ ]:
title = []

for p in pids:
    get_dataset_metadata_url = "%s/api/datasets/:persistentId/?persistentId=%s" % (base_url, p)
    r = requests.get(get_dataset_metadata_url, headers=headers)
    title.append(r.json()['data']['latestVersion']["metadataBlocks"]["citation"]["fields"][0]["value"])

pids_zip = list(zip(pids, title))
doi_df = pd.DataFrame(pids_zip, columns=['pid','title'])

In [ ]:
doi_df

In [ ]:
for index, row in doi_df.iterrows():
    k = row['title']
    filename = k.replace(".JPG", "")
    print('2012/%s.json' % (filename))

In [ ]:
for index, row in doi_df.iterrows():
    k = row['title']
    filename = k.replace(".JPG", "")
    filepath = 'test/%s.json' % (filename)
    print(filepath)

In [ ]:
for index, row in doi_df.iterrows():
    k = row['title']
    filename = k.replace(".JPG", "")
    filepath = '2012/%s.json' % (filename)
    edit_metadata_url = "%s/api/datasets/:persistentId/editMetadata/?persistentId=%s&replace=true" % (base_url, row['pid'])
    r = requests.put(edit_metadata_url, headers=headers, data=open(filepath, "r").read())
    print("Dataset" + k + "edited" + r.json()['status'])

### 8. Find datasets with missing files

`for doi in list_of_dois: 
     get files
     if r.json()[data] != has file
         empty_dois.append(doi)`

In [ ]:
import pandas as pd
import json
import requests
import itertools

api_token = {api_token}
base_url = "https://dataverse.harvard.edu"

headers = {'X-Dataverse-key': api_token,}

In [ ]:
#print dois from step 6
len(pids)

In [ ]:
missing_files = []

for pid in pids:
    list_files_url = "%s/api/datasets/:persistentId/versions/:draft/files?persistentId=%s" % (base_url, pid)
    r = requests.get(list_files_url, headers = headers)
    if r.json()['data'] == []:
        missing_files.append(pid)
        continue

In [ ]:
len(missing_files)

In [ ]:
with open("2012/empty_datasets.txt", "w+") as f:
    for item in missing_files:
        f.write("%s\n" % (item))

f.close()

### 9. Get filenames of missing images

In [ ]:
file = open("2012/empty_datasets.txt", "r")
pids = file.readlines()
file.close()

In [ ]:
pids_stripped = []
for t in pids:
    pids_stripped.append(t.rstrip())

pids_stripped

In [ ]:
api_token = {api_token}
base_url = "https://dataverse.harvard.edu"

headers = {'X-Dataverse-key': api_token,}

In [ ]:
title = []

for p in pids_stripped:
    get_dataset_metadata_url = "%s/api/datasets/:persistentId/?persistentId=%s" % (base_url, p)
    r = requests.get(get_dataset_metadata_url, headers=headers)
    if(r.json()['status']== "OK"):
        title.append(r.json()['data']['latestVersion']["metadataBlocks"]["citation"]["fields"][0]["value"])
        continue

pids_zip = list(zip(pids, title))
doi_df = pd.DataFrame(pids_zip, columns=['pid','title'])

In [ ]:
doi_df = doi_df.replace(r'\n',' ', regex=True) 
doi_df

`for doi in doi_df:
    find file in 2021_images directory
    upload file`

In [ ]:
doi_df.to_csv('2012_images/datasets.csv', index = False)

In [ ]:
import os
from pyDataverse.utils import read_csv_as_dicts

In [ ]:
csv_datafiles_filename = "2012_images/levy_datafiles_2012_missing.csv"
df_data = read_csv_as_dicts(csv_datafiles_filename)

In [ ]:
from pyDataverse.models import Datafile
df_lst = []
for df in df_data:
    df_obj = Datafile()
    df_obj.set(df)
    df_lst.append(df_obj)

In [ ]:
from pyDataverse.api import NativeApi
api = NativeApi(base_url, api_token)

In [ ]:
path = "/Users/katherinemika/Desktop/curation/leon_levy/2012_images"
for df in df_lst:
    pid = df.get()['org.pid']
    filename = os.path.join(path, df.get()['org.filename'])
    df.set({"pid": pid, "filename": filename})
    resp = api.upload_datafile(pid, filename, df.json())

In [ ]:
with open("2012/2012_dois.txt", "r") as f:
    dois = f.readlines()
    
dois_stripped = []
for t in dois:
    dois_stripped.append(t.rstrip())

dois_stripped

In [ ]:
#find datasets with duplicate data
for doi in dois_stripped:
    url = "%s/api/datasets/:persistentId/versions/:draft/files?persistentId=%s" % (base_url, doi)
    r = requests.get(url, headers=headers)
    try:
        print(r.json()['data'][1])
    except:
        pass

### 10. Find missing datasets

In [ ]:
import pandas as pd
import json
import requests
import itertools

api_token = {api_token}
base_url = "https://dataverse.harvard.edu"

headers = {'X-Dataverse-key': api_token,}
titles = []

In [ ]:
#2010
titles_2010 = []

dv_url = "%s/api/dataverses/levy_photos_2010/contents" % (base_url)
r = requests.get(dv_url, headers=headers)
json_2010 = r.json()

id_2010 = []
for i in json_2010['data']:
    id_2010.append(i['id'])

for j in id_2010:
    url = "%s/api/datasets/%s" % (base_url, j)
    r = requests.get(url, headers=headers)
    titles_2010.append(r.json()['data']['latestVersion']['metadataBlocks']['citation']['fields'][0]['value'])

In [ ]:
titles_2010

In [ ]:
#2010-2017
dv_url = "%s/api/dataverses/levy_photos_2017/contents" % (base_url)
r = requests.get(dv_url, headers=headers)
json_2017 = r.json()

id_2017 = []
for i in json_2017['data']:
    id_2017.append(i['id'])

for j in id_2017:
    url = "%s/api/datasets/%s" % (base_url, j)
    r = requests.get(url, headers=headers)
    titles.append(r.json()['data']['latestVersion']['metadataBlocks']['citation']['fields'][0]['value'])

In [ ]:
len(titles)

In [ ]:
with open("titles_10_17.txt", "w") as f:
    for  title in titles:
        f.write("%s\n" % (title))
        
f.close()

In [ ]:
#find which titles are missing 
with open("titles_10_17.csv", "r") as f:
    u = f.readlines()
f.close()

uploaded = []
for t in u:
    uploaded.append(t.rstrip())

uploaded = pd.DataFrame(uploaded)
uploaded = uploaded.rename(columns = {0:"filename"})
uploaded

In [ ]:
master = pd.read_csv("masterlist_10_17.csv")
master

In [ ]:
#find rows in master that have no master["Filename"] match in uploaded
common = master.merge(uploaded, on=['filename'])
result = master[~master.filename.isin(common.filename)]
result

In [ ]:
result.reset_index()

In [ ]:
#save list of missing datasets
result.to_csv("missing_data/missing_datasets_3.csv", index = False)

### 11. Redetect file types

In [ ]:
import requests
import pandas as pd
import itertools
from time import sleep
import os
import json

In [ ]:
api_token = {api_token}
base_url = "https://dataverse.harvard.edu"


headers = {'X-Dataverse-key': api_token,}

In [ ]:
download_links = pd.read_csv("1985_redetect.csv")
download_links.head()

In [ ]:
download_links['fileId'] = download_links.link.str[50:]
download_links.head()

In [ ]:
responses = []
for index, row in download_links.iterrows():
    url = "%s/api/files/%s/redetect" % (base_url, row['fileId'])
    r = requests.post(url, headers=headers)
    responses.append(r.json())

In [ ]:
responses

## 12. Edit File Metadata

In [ ]:
import requests
import pandas as pd
import itertools
from time import sleep
import os
import json

api_token = {api_token}
base_url = "https://dataverse.harvard.edu"

headers = {'X-Dataverse-key': api_token, }
headers_post_json = {'X-Dataverse-key': api_token, "content-type":"application/json"}

In [ ]:
#get file id from dataset json
ds_json_url = "%s/api/datasets/:persistentId/?persistentId=doi:10.70122/FK2/880VYJ" % (base_url)
r = requests.get(ds_json_url, headers=headers)
response = r.json()

In [ ]:
response['data']['latestVersion']['files']

In [ ]:
#get file metadata to edit
df_json_url = "%s/api/files/1955593/metadata" % (base_url)
r = requests.get(df_json_url, headers=headers)
print(r.json())

In [ ]:
#load metadata file
with open("json_data_2012.json", "rb") as f:
    dat = json.load(f)
    
print(dat)

In [ ]:
#jsonData = {'description': 'http://pi.lib.uchicago.edu/1001/org/ochre/bdc226bb-d503-4fa3-be56-8f290383cba8'}
Data={"description":"description number three!","categories":["Illustration"],"restrict":"false"}

In [ ]:
type(Data)

In [ ]:
jsonData = json.dumps(Data)

In [ ]:
type(jsonData)

In [ ]:
data={"description":"My new description","categories":["Photo"], "restrict":"false"}
headers = {'X-Dataverse-key': api_token, }

update_file_metadata_url = "%s/api/files/1955611/metadata" % (base_url)
r = requests.post(update_file_metadata_url, data = data, headers=headers_post_json)
print(r)

In [ ]:
#pydataverse
from pyDataverse.api import *
from pyDataverse.api import NativeApi

apikey = 'xxx'
api = NativeApi('https://demo.dataverse.org', api_token=apikey)
file_id = "1955611"

json_str = '{"description":"new description", "categories":["Photo"]}'

r = api.update_datafile_metadata(file_id, json_str=json_str, is_filepid=False)

print(r)

## Use Ceilyn's function to edit files missing descriptions and tags

1. Use the list of FileIds from the csv files to find ones that are missing descriptions
2. Get the missing descriptions from the ochre spreadsheet using the filenames
3. for each file ID, update metadata with correct description and tag

In [ ]:
#read in big list of all files
all_files = pd.read_csv("photographs_deposited_full.csv")
all_files.head()

In [ ]:
remainders = pd.read_csv("remainders.csv")
remainders.head()

In [ ]:
#Create list of files missing descriptions (filename, fileId) - MUST BE PUBLISHED VERSION! WILL GIVE 500 ERROR WITHOUGHT PASSING "/DRAFT" TO THE END OF THE URL IF THE DATASET IS IN DRAFT STATUS
#edit_desc = pd.DataFrame(columns = ['filename', 'doi','fileId']) #comment out after initialized (may need to run this block a few times...)
for index, ds in remainders.iterrows():
    url = "%s/api/files/%s/metadata" % (base_url, ds['file_id'])
    r = requests.get(url, headers=headers)
    try:
        r.json()['description']
    except KeyError:
        filename = r.json()['label']
        doi = ds['doi']
        fileId = ds['file_id']
        temp_dict = {'filename':filename, 'doi':doi, 'fileId':fileId}
        edit_desc = edit_desc.append(temp_dict, ignore_index=True)

In [ ]:
edit_desc

In [ ]:
edit_desc.to_csv("to_edit_descriptions.csv", index = False)

In [ ]:
edit = pd.read_csv("to_edit_descriptions.csv")
metadata = pd.read_csv("levy_ochre_links.csv")
metadata.head()

In [ ]:
#get links for each file in edit_desc
toedit = metadata.merge(edit, on = ['filename'])
toedit

In [ ]:
#update dataset metadata function (from CB) to generate a draft that can then be updated with the file metadata changes
import requests
from pyDataverse.api import NativeApi

base_url = "https://dataverse.harvard.edu"
api_token = {api_token}
api = NativeApi(base_url, api_token)

def edit_dataset_metadata(api, dataset_doi):
    # get the dataset metadata using pyDataverse
    response = api.get_dataset(dataset_doi, version=':latest', auth=True, is_pid=True)
    status = response.json().get('status')
    
    # if the call to get dataset metadata failed, signal error and return
    if (status != 'OK'):
        print('edit_dataset_metadata: Failed to get metadata for {}, failed with {}'.format(dataset_doi, status))
        return status
    
    # get the metadata
    md = response.json().get('data').get('latestVersion').get('metadataBlocks').get('citation')
    # get the first field from the dataset metadata
    field = md['fields'][0]
    # create an edited version of this field
    fields = {'fields':[field]}
    
    #
    # prepare to call the requests library to edit the dataset (thereby creating a draft)
    #
    
    # get the base url
    base_url = api.base_url
    # get the api token
    api_token = api.api_token
    # create the headers
    headers = {'X-Dataverse-key': api_token, 'Content-Type' : 'application/json'}
    # create the request url, note the replace=true parameter
    request_url = '{}/api/datasets/:persistentId/editMetadata/?persistentId={}&replace=true'.format(base_url, dataset_doi)
    
    # call the requests library using the request url
    response = requests.put(request_url, headers=headers, data=json.dumps(fields))
    # handle responses
    status = response.json().get('status')
    if (status != 'OK'):
        print('edit_dataset_metadata: Error - failed to edit metadata on {}'.format(dataset_doi))
        return response.json()
    # otherwise, return 
    return status

In [ ]:
for index, ds in toedit.iterrows():
    edit_dataset_metadata(api, ds['doi'])

In [ ]:
#edit file metadata
from pyDataverse.api import *
from pyDataverse.api import NativeApi

base_url = "https://dataverse.harvard.edu"
api_token = {api_token}
api = NativeApi(base_url, api_token)

for index, ds in toedit.iterrows():
    file_id = ds['fileId']
    json_str = '{"description":"' + ds['ochre_metadata'] + '", "categories":["Photo"]}'
    r = api.update_datafile_metadata(file_id, json_str=json_str, is_filepid=False)

In [ ]:
#publish edited drafts
MAJOR_OR_MINOR = "MAJOR"
for index, ds in toedit.iterrows():
    publish = "%s/api/datasets/:persistentId/actions/:publish?persistentId=%s&type=%s" % (base_url, ds['doi'], MAJOR_OR_MINOR)
    r = requests.post(publish, headers=headers)
    sleep(30)

In [ ]:
toedit


## -------TESTING------------------TESTING-------------------TESTING------------TESTING---------

In [ ]:
f = open("test_dois_edit.txt", "r")
dois_test = f.read().splitlines()

In [ ]:
import pandas as pd
import json
import requests
import itertools

api_token = '46d0ba22-f34a-4a03-94fd-98a143931586'
base_url = "https://demo.dataverse.org"

headers = {'X-Dataverse-key': api_token,}

In [ ]:
list_files_url = "%s/api/datasets/:persistentId/versions/:draft/files?persistentId=doi:10.70122/FK2/GTMPX2" % (base_url)
r = requests.get(list_files_url, headers=headers)
print(r.json()['data'])

In [ ]:
missing_files = []

for doi in dois_test:
    list_files_url = "%s/api/datasets/:persistentId/versions/:draft/files?persistentId=%s" % (base_url, doi)
    r = requests.get(list_files_url, headers=headers)
    if r.json()['data'] == []:
        missing_files.append(doi)
        continue

In [ ]:
missing_files

In [ ]:
edit_metadata_url = "https://demo.dataverse.org/api/datasets/:persistentId/editMetadata/?persistentId=doi:10.70122/FK2/GTMPX2&replace=true"
filepath = "test/A17_33794.json"
r = requests.put(edit_metadata_url, headers=headers, data = open(filepath, 'r').read())
print(r.text)

In [ ]:
get_dataset_metadata_url = "%s/api/datasets/:persistentId/?persistentId=doi:10.7910/DVN/GT6V1D" % (base_url)
r = requests.get(get_dataset_metadata_url, headers=headers)
print(r.json()['data']['latestVersion']["metadataBlocks"]["citation"]["fields"][0]["value"])

In [ ]:
for df in df_lst:
    print([df.get()['org.dataset_id']])

In [ ]:
dois

In [ ]:
d = {str(key): str(value) for key, value in dois.items()}
print(d)

In [ ]:
for df in df_lst:
    print(d[df.get()['org.dataset_id']])

In [ ]:
export API_TOKEN={api_token}
export SERVER_URL=https://dataverse.harvard.edu
export PERSISTENT_IDENTIFIER=doi:10.7910/DVN/SOFHNB

curl -H "X-Dataverse-key: $API_TOKEN" -X PUT $SERVER_URL/api/datasets/:persistentId/editMetadata/?persistentId=$PERSISTENT_IDENTIFIER --upload-file levy_edit_grant.json



In [ ]:
text_file = open('test_dois.txt', 'r')
test_dois = text_file.readlines()
print(test_dois)

In [ ]:
test_dois[0]

In [ ]:
strip_dois = []
for i in test_dois:
    strip_dois.append(i.rstrip())
    

In [ ]:
strip_dois

In [ ]:
headers = {'X-Dataverse-key': api_token,}

for i in strip_dois:
    url_dataset_move = '%s/api/datasets/:persistentId/ll_test?persistentId=%s' % (base_url, i)
    print(url_dataset_move)

In [ ]:
headers = {'X-Dataverse-key': api_token,}

for i in strip_dois:
    url_dataset_move = '%s/api/datasets/:persistentId/move/ll_test?persistentId=%s' % (base_url, i)
    r = requests.post(url_dataset_move, headers=headers)
    print(r.json())

In [ ]:
curl -H "X-Dataverse-key: $API_TOKEN" -X POST $SERVER/api/datasets/:persistentId/move/$alias?persistentId=$PERSISTENT_ID